# Isopropyl Alcohol

In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
isopropylalcohol,60.096,2.8955,3.270409011,209.6466379,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
isopropylalcohol,H,isopropylalcohol,e,2000.0,0.03
"""

model = PCSAFT(["isopropylalcohol"], userlocations = [like_parameter, assoc_parameter])
print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2000.0]

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_isopropylalcohol.csv")
fix_line_endings("rhol_isopropylalcohol.csv")

Fixed: satp_isopropylalcohol.csv
Fixed: rhol_isopropylalcohol.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

println(my_liquid_density(model, 293.15))
println(my_saturation_p(model, 273.15))

780.1464468243072
2911.6432761789033


In [5]:
toestimate = [
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 1.0,
        :guess   => 0.4
    )
]

2-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 0.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.0)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_isopropylalcohol.csv",
        "satp_isopropylalcohol.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 0.8701990522699787


In [7]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([2372.983674759261, 0.02184505149554316], PCSAFT{BasicIdeal, Float64}("isopropylalcohol"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_isopropylalcohol.csv",   my_saturation_p)


=== AAD: satp_isopropylalcohol.csv ===

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593



Clapeyron Estimator  exp           calc          ARD%    
305.7800    9560.000000   9628.124105   0.7126  
308.5700    11320.000000  11269.950350  0.4421  
311.1200    12900.000000  12975.042934  0.5817  
314.0200    15270.000000  15177.984448  0.6026  
316.9700    17780.000000  17738.771294  0.2319  
319.6200    20470.000000  20343.958897  0.6157  
322.4300    23350.000000  23454.290655  0.4466  
324.8900    26590.000000  26498.945480  0.3424  
327.3400    29750.000000  29856.479520  0.3579  
330.3500    34430.000000  34465.595430  0.1034  
333.6700    40340.000000  40228.431103  0.2766  
339.2600    51700.000000  51748.466667  0.0937  
342.4100    59250.000000  59369.380211  0.2015  
345.8400    68840.000000  68704.496283  0.1968  
348.2300    76090.000000  75901.834646  0.2473  
350.7500    83990.000000  84153.407871  0.1946  
352.1900    89190.000000  89191.018363  0.0011  
355.4100    101300.000000  101356.517104  0.0558  
355.4200    101220.000000  101396.312653  0.1742  
AARD =

0.3094030593082032

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_isopropylalcohol.csv",   my_liquid_density)


=== AAD: rhol_isopropylalcohol.csv ===
Clapeyron Estimator  exp           calc          ARD%    


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


293.1500    785.400000    784.726928    0.0857  
298.1500    781.100000    780.445023    0.0839  
303.1500    776.800000    776.121716    0.0873  
308.1500    772.400000    771.751807    0.0839  
313.1500    768.000000    767.330068    0.0872  
318.1500    763.400000    762.851221    0.0719  
323.1500    758.800000    758.309921    0.0646  
AARD = 0.0806%


0.08064157346093218